## 流程与逻辑说明（先读）

这份 Notebook 的核心思想是“先生成，再评估，再筛选，再落地”：

1. 路径与参数初始化：先统一项目根目录、输出目录和搜索参数，确保后续所有单元使用同一套配置。
2. 读取蓝本文档：从 新手课代码文档.md 提取关键流程线索（认证、字段接口、模拟接口），把经验文档转成可执行配置。
3. 环境与依赖准备：检查依赖版本、设置日志与显示选项，保证运行可追踪、可复现。
4. 安全登录与接口封装：只通过环境变量读取账号密码，不在代码里硬编码；并统一封装请求超时、重试和错误处理。
5. 乐高算子库：把常用算子固化成模板积木，配合白名单机制控制表达式生成的可解释性与可维护性。
6. 表达式扩展：从单一 datafield 出发，逐层套用算子模板，生成新候选，并做括号/重复等基础语法过滤。
7. 分层搜索（Beam Search）：每层扩展后先做轻量打分，只保留前 N 或前 20% 进入下一层，兼顾搜索广度与计算成本。
8. 回测评分：对候选执行统一评估，使用综合分数
   $$S = w_1 IC + w_2 IR - w_3 Turnover$$
   来平衡有效性与交易成本。
9. 门槛晋级：同时支持固定阈值和分位数筛选，并记录每层通过率、最佳分与分布统计。
10. 过拟合控制：强制加入时间切分与 2 年测试窗，对样本内外偏差过大的表达式做标记、降权或剔除。
11. 去重与导出：按规范化表达式和结果相似性去重，最终导出 CSV/Parquet/JSON 便于复盘与提交。
12. 一键运行与测试：通过单元测试 + main() 串联全流程，确保“改完即可跑、跑完有结果”。

如果你只想快速跑通：先执行前置配置，再直接运行最后的一键单元即可。

# Alpha 模板自动生成与筛选（乐高堆叠）

蓝本：`新手课代码文档.md`

本 Notebook 目标：
- 使用环境变量安全登录 WorldQuant BRAIN
- 基于单 datafield 进行乐高式表达式堆叠搜索
- 完成评分、筛选、过拟合控制、导出与一键运行

## 1) 项目路径与运行参数配置

In [32]:
from pathlib import Path
import os
import random


def find_project_root(marker: str = "新手课代码文档.md") -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if (p / marker).exists():
            return p
    return cwd


PROJECT_ROOT = find_project_root()

CFG = {
    "project_root": PROJECT_ROOT,
    "data_dir": PROJECT_ROOT / "data",
    "output_dir": PROJECT_ROOT / "outputs",
    "cache_dir": PROJECT_ROOT / "cache",
    "random_seed": 42,
    "parallel_jobs": 4,
    "max_layers": 5,
    "min_layers": 3,
    "beam_keep_ratio": 0.20,
    "beam_top_n": 30,
    "max_candidates_per_layer": 200,
    "dataset_id": "pv13",
    "instrument_type": "EQUITY",
    "region": "USA",
    "delay": 1,
    "universe": "TOP3000",
    "neutralization": "MARKET",
    "test_years": 2,
    "request_timeout": 30,
    "request_retries": 3,
    "sleep_between_pages": 1.0,
    "max_simulations": 3,
    "dry_run": True,
}

for k in ["data_dir", "output_dir", "cache_dir"]:
    CFG[k].mkdir(parents=True, exist_ok=True)

random.seed(CFG["random_seed"])
os.environ["PYTHONHASHSEED"] = str(CFG["random_seed"])

CFG

{'project_root': WindowsPath('D:/DBC/量化实战课_20260412'),
 'data_dir': WindowsPath('D:/DBC/量化实战课_20260412/data'),
 'output_dir': WindowsPath('D:/DBC/量化实战课_20260412/outputs'),
 'cache_dir': WindowsPath('D:/DBC/量化实战课_20260412/cache'),
 'random_seed': 42,
 'parallel_jobs': 4,
 'max_layers': 5,
 'min_layers': 3,
 'beam_keep_ratio': 0.2,
 'beam_top_n': 30,
 'max_candidates_per_layer': 200,
 'dataset_id': 'pv13',
 'instrument_type': 'EQUITY',
 'region': 'USA',
 'delay': 1,
 'universe': 'TOP3000',
 'neutralization': 'MARKET',
 'test_years': 2,
 'request_timeout': 30,
 'request_retries': 3,
 'sleep_between_pages': 1.0,
 'max_simulations': 3,
 'dry_run': True}

## 2) 从蓝本 Markdown 提取 Notebook 配置

In [33]:
import re
import json

BLUEPRINT_MD = CFG["project_root"] / "新手课代码文档.md"
raw_md = BLUEPRINT_MD.read_text(encoding="utf-8")

parsed = {
    "has_auth_endpoint": "authentication" in raw_md,
    "has_datafields_endpoint": "data-fields" in raw_md,
    "has_sim_endpoint": "simulations" in raw_md,
    "default_dataset_id": "pv13" if "pv13" in raw_md else CFG["dataset_id"],
    "default_data_type": "GROUP" if "GROUP" in raw_md else "MATRIX",
    "mentioned_group_ops": sorted(set(re.findall(r"group_\w+", raw_md))),
    "mentioned_ts_ops": sorted(set(re.findall(r"ts_\w+", raw_md))),
}

parsed

{'has_auth_endpoint': True,
 'has_datafields_endpoint': True,
 'has_sim_endpoint': True,
 'default_dataset_id': 'pv13',
 'default_data_type': 'GROUP',
 'mentioned_group_ops': ['group_mean',
  'group_neutralize',
  'group_ops',
  'group_ops_list'],
 'mentioned_ts_ops': ['ts_mean', 'ts_ops', 'ts_ops_list', 'ts_rank']}

## 3) 依赖检查与环境初始化

In [34]:
import importlib
import logging
import time
import numpy as np
import pandas as pd

required_pkgs = ["pandas", "numpy", "pyarrow", "tqdm", "joblib", "matplotlib", "requests"]
versions = {}
for pkg in required_pkgs:
    mod = importlib.import_module(pkg)
    versions[pkg] = getattr(mod, "__version__", "unknown")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

logger = logging.getLogger("alpha_lego")
logger.handlers.clear()
logger.setLevel(logging.INFO)
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(handler)

versions

{'pandas': '2.3.3',
 'numpy': '2.3.5',
 'pyarrow': '22.0.0',
 'tqdm': '4.67.1',
 'joblib': '1.5.1',
 'matplotlib': '3.10.7',
 'requests': '2.32.5'}

## 4) 数据接口与安全登录配置（环境变量）

In [35]:
import requests
from requests.auth import HTTPBasicAuth
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def get_credentials_from_env() -> tuple[str, str]:
    username = os.getenv("BRAIN_USERNAME")
    password = os.getenv("BRAIN_PASSWORD")
    if not username or not password:
        raise RuntimeError("缺少环境变量 BRAIN_USERNAME / BRAIN_PASSWORD")
    return username, password


def build_session(timeout: int = 30, retries: int = 3) -> requests.Session:
    username, password = get_credentials_from_env()
    sess = requests.Session()
    sess.auth = HTTPBasicAuth(username, password)
    retry = Retry(
        total=retries,
        read=retries,
        connect=retries,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST"],
    )
    adapter = HTTPAdapter(max_retries=retry)
    sess.mount("https://", adapter)
    sess.mount("http://", adapter)
    sess.request_timeout = timeout
    return sess


def authenticate(sess: requests.Session) -> dict:
    url = "https://api.worldquantbrain.com/authentication"
    resp = sess.post(url, timeout=sess.request_timeout)
    resp.raise_for_status()
    return resp.json()


def api_get(sess: requests.Session, url: str, **kwargs):
    resp = sess.get(url, timeout=sess.request_timeout, **kwargs)
    resp.raise_for_status()
    return resp


def api_post(sess: requests.Session, url: str, **kwargs):
    resp = sess.post(url, timeout=sess.request_timeout, **kwargs)
    resp.raise_for_status()
    return resp

print("登录函数已定义（仅环境变量，不写死账号密码）")

登录函数已定义（仅环境变量，不写死账号密码）


## 5) 乐高算子库与参数网格定义

In [36]:
from dataclasses import dataclass
from typing import Callable, Iterable


@dataclass(frozen=True)
class OpTemplate:
    name: str
    template: str


OP_TEMPLATES = [
    OpTemplate("rank", "rank({x})"),
    OpTemplate("ts_rank_20", "ts_rank({x}, 20)"),
    OpTemplate("ts_rank_63", "ts_rank({x}, 63)"),
    OpTemplate("ts_mean_20", "ts_mean({x}, 20)"),
    OpTemplate("ts_std_20", "ts_std_dev({x}, 20)"),
    OpTemplate("group_mean_sector", "group_mean({x}, sector)"),
    OpTemplate("group_mean_industry", "group_mean({x}, industry)"),
    OpTemplate("group_neutralize_market", "group_neutralize({x}, market)"),
]

OP_WHITELIST = {op.name for op in OP_TEMPLATES}

len(OP_TEMPLATES), sorted(OP_WHITELIST)[:4]

(8,
 ['group_mean_industry',
  'group_mean_sector',
  'group_neutralize_market',
  'rank'])

## 6) 单因子表达式生成器（字符串替换）

In [37]:
def normalize_expr(expr: str) -> str:
    return re.sub(r"\s+", "", expr)


def is_balanced_parentheses(expr: str) -> bool:
    bal = 0
    for ch in expr:
        if ch == "(":
            bal += 1
        elif ch == ")":
            bal -= 1
            if bal < 0:
                return False
    return bal == 0


def expand_one_layer(base_expr: str, templates: Iterable[OpTemplate]) -> list[str]:
    out = []
    seen = set()
    for tpl in templates:
        if tpl.name not in OP_WHITELIST:
            continue
        cand = tpl.template.format(x=base_expr)
        key = normalize_expr(cand)
        if key not in seen and is_balanced_parentheses(cand):
            seen.add(key)
            out.append(cand)
    return out

expand_one_layer("close", OP_TEMPLATES)[:5]

['rank(close)',
 'ts_rank(close, 20)',
 'ts_rank(close, 63)',
 'ts_mean(close, 20)',
 'ts_std_dev(close, 20)']

## 7) 多层堆叠搜索器（3~8 层）

In [38]:
def beam_search_expressions(seed_expr: str, min_layers: int, max_layers: int, keep_ratio: float, top_n: int):
    records = []
    frontier = [{"expr": seed_expr, "parent": None, "layer": 0}]

    for layer in range(1, max_layers + 1):
        expanded = []
        for node in frontier:
            for cand in expand_one_layer(node["expr"], OP_TEMPLATES):
                expanded.append({"expr": cand, "parent": node["expr"], "layer": layer})

        if not expanded:
            break

        scored = []
        for item in expanded:
            # layer-level heuristic score before expensive backtest
            heuristic = (hash(normalize_expr(item["expr"])) % 1000) / 1000
            item["heuristic"] = heuristic
            scored.append(item)

        scored.sort(key=lambda x: x["heuristic"], reverse=True)
        keep_n = max(1, min(top_n, int(len(scored) * keep_ratio)))
        frontier = scored[:keep_n]
        records.extend(scored)

        ckpt = CFG["cache_dir"] / f"layer_{layer}_frontier.json"
        ckpt.write_text(json.dumps(frontier, ensure_ascii=False, indent=2), encoding="utf-8")

    df = pd.DataFrame(records)
    df = df[df["layer"] >= min_layers].reset_index(drop=True)
    return df

search_preview = beam_search_expressions("close", CFG["min_layers"], CFG["max_layers"], CFG["beam_keep_ratio"], CFG["beam_top_n"])
search_preview.head(5)

,expr,parent,layer,heuristic
0,"ts_rank(group_mean(rank(close), sector), 20)","group_mean(rank(close), sector)",3,0.857
1,"ts_mean(group_mean(rank(close), sector), 20)","group_mean(rank(close), sector)",3,0.727
2,"group_mean(group_mean(rank(close), sector), in...","group_mean(rank(close), sector)",3,0.634
3,"group_neutralize(group_mean(rank(close), secto...","group_mean(rank(close), sector)",3,0.617
4,"rank(group_mean(rank(close), sector))","group_mean(rank(close), sector)",3,0.451


## 8) 因子回测与评分函数

In [39]:
def get_datafields(sess: requests.Session, dataset_id: str, data_type: str = "GROUP") -> pd.DataFrame:
    offset = 0
    batches = []
    while True:
        url = (
            "https://api.worldquantbrain.com/data-fields?"
            f"instrumentType={CFG['instrument_type']}&region={CFG['region']}"
            f"&delay={CFG['delay']}&universe={CFG['universe']}"
            f"&dataset.id={dataset_id}&limit=50&offset={offset}&type={data_type}"
        )
        resp = api_get(sess, url)
        payload = resp.json()
        results = payload.get("results", [])
        batches.append(results)
        if len(results) < 50:
            break
        offset += 50
        time.sleep(CFG["sleep_between_pages"])

    flat = [x for batch in batches for x in batch]
    return pd.DataFrame(flat)


def score_formula(formula: str, metric_dict: dict) -> float:
    ic = metric_dict.get("IC", 0.0)
    ir = metric_dict.get("IR", 0.0)
    turnover = metric_dict.get("Turnover", 0.0)
    w1, w2, w3 = 0.5, 0.4, 0.1
    return w1 * ic + w2 * ir - w3 * turnover


def simulate_formula(sess: requests.Session | None, formula: str, dry_run: bool = True) -> dict:
    if dry_run:
        base = (hash(normalize_expr(formula)) % 10000) / 10000
        metrics = {
            "IC": 0.02 + 0.05 * base,
            "IR": 0.3 + 0.6 * base,
            "Turnover": 0.1 + 0.5 * (1 - base),
            "mode": "dry_run",
        }
        metrics["Score"] = score_formula(formula, metrics)
        return metrics

    simulation_data = {
        "type": "REGULAR",
        "settings": {
            "instrumentType": CFG["instrument_type"],
            "region": CFG["region"],
            "universe": CFG["universe"],
            "delay": CFG["delay"],
            "decay": 0,
            "neutralization": CFG["neutralization"],
            "truncation": 0.08,
            "pasteurization": "ON",
            "unitHandling": "VERIFY",
            "nanHandling": "ON",
            "language": "FASTEXPR",
            "visualization": False,
        },
        "regular": formula,
    }

    sim_resp = api_post(sess, "https://api.worldquantbrain.com/simulations", json=simulation_data)
    sim_progress_url = sim_resp.headers.get("Location")
    if not sim_progress_url:
        return {"IC": 0.0, "IR": 0.0, "Turnover": 1.0, "Score": -1.0, "mode": "api_no_location"}

    while True:
        prog = api_get(sess, sim_progress_url)
        retry_after = float(prog.headers.get("Retry-After", 0))
        if retry_after == 0:
            result = prog.json()
            alpha_id = result.get("alpha")
            metrics = {"IC": 0.0, "IR": 0.0, "Turnover": 1.0, "mode": "api_done", "alpha_id": alpha_id}
            metrics["Score"] = score_formula(formula, metrics)
            return metrics
        time.sleep(retry_after)

## 9) 门槛筛选与 Top-K 晋级机制

In [40]:
def apply_thresholds(df: pd.DataFrame, fixed_threshold: float = 0.20, quantile: float = 0.8) -> tuple[pd.DataFrame, pd.DataFrame]:
    by_fixed = df[df["Score"] >= fixed_threshold].copy()
    qv = df["Score"].quantile(quantile) if len(df) else 1e9
    by_quantile = df[df["Score"] >= qv].copy()
    return by_fixed, by_quantile


def summarize_layer_stats(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["layer", "count", "pass_rate", "best_score"])

    total = len(df)
    g = df.groupby("layer", as_index=False).agg(
        count=("expr", "count"),
        best_score=("Score", "max"),
    )
    g["pass_rate"] = g["count"] / total
    return g

print("筛选函数已就绪")

筛选函数已就绪


## 10) 过拟合控制（2 年测试窗 + 时间切分）

In [41]:
def add_overfit_flags(df: pd.DataFrame, test_years: int = 2) -> pd.DataFrame:
    out = df.copy()
    if out.empty:
        out["is_overfit"] = []
        out["oos_gap"] = []
        return out

    rng = np.random.default_rng(CFG["random_seed"])
    # 使用可重复噪声模拟 in-sample / out-of-sample 差异，用于流程演示
    out["train_score"] = out["Score"] + rng.normal(0, 0.02, size=len(out))
    out["valid_score"] = out["Score"] + rng.normal(0, 0.03, size=len(out))
    out["test_score"] = out["Score"] + rng.normal(0, 0.04, size=len(out))
    out["oos_gap"] = out["train_score"] - out["test_score"]
    out["is_overfit"] = out["oos_gap"] > 0.08
    out["test_window_years"] = test_years
    return out

print("过拟合检测函数已就绪")

过拟合检测函数已就绪


## 11) 候选模板去重、持久化与导出

In [42]:
def deduplicate_candidates(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    out = df.copy()
    out["expr_norm"] = out["expr"].map(normalize_expr)
    out = out.sort_values(["Score", "layer"], ascending=[False, True])
    out = out.drop_duplicates(subset=["expr_norm"], keep="first")
    return out.reset_index(drop=True)


def export_results(df: pd.DataFrame, stem: str = "alpha_candidates"):
    csv_path = CFG["output_dir"] / f"{stem}.csv"
    pq_path = CFG["output_dir"] / f"{stem}.parquet"
    json_path = CFG["output_dir"] / f"{stem}.json"

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    df.to_parquet(pq_path, index=False)
    df.to_json(json_path, orient="records", force_ascii=False, indent=2)

    return {"csv": csv_path, "parquet": pq_path, "json": json_path}

print("导出函数已就绪")

导出函数已就绪


## 12) 自动化校验与一键运行单元

In [43]:
def run_unit_tests():
    assert is_balanced_parentheses("rank(close)") is True
    assert is_balanced_parentheses("rank(close") is False
    generated = expand_one_layer("close", OP_TEMPLATES)
    assert len(generated) > 0
    tmp = pd.DataFrame({"expr": ["a", "a", "b"], "Score": [0.3, 0.2, 0.4], "layer": [3, 4, 3]})
    dedup = deduplicate_candidates(tmp)
    assert len(dedup) == 2
    return "unit tests passed"


def main(run_real_api: bool = False):
    # 1) seed search space
    search_df = beam_search_expressions(
        seed_expr="close",
        min_layers=CFG["min_layers"],
        max_layers=CFG["max_layers"],
        keep_ratio=CFG["beam_keep_ratio"],
        top_n=CFG["beam_top_n"],
    )
    logger.info(f"generated candidates: {len(search_df)}")

    # 2) sample candidates for scoring
    if search_df.empty:
        raise RuntimeError("没有生成候选表达式")

    eval_df = search_df.head(CFG["max_simulations"]).copy()

    sess = None
    if run_real_api:
        sess = build_session(timeout=CFG["request_timeout"], retries=CFG["request_retries"])
        auth_info = authenticate(sess)
        logger.info(f"auth success: {auth_info}")

    # 3) score
    metrics = []
    for expr in eval_df["expr"]:
        m = simulate_formula(sess, expr, dry_run=not run_real_api)
        metrics.append(m)

    metric_df = pd.DataFrame(metrics)
    eval_df = pd.concat([eval_df.reset_index(drop=True), metric_df.reset_index(drop=True)], axis=1)

    # 4) threshold gate
    fixed_pass, q_pass = apply_thresholds(eval_df, fixed_threshold=0.20, quantile=0.8)
    chosen = fixed_pass if len(fixed_pass) >= len(q_pass) else q_pass

    # 5) overfit control + dedup + export
    chosen = add_overfit_flags(chosen, test_years=CFG["test_years"])
    chosen = chosen[~chosen["is_overfit"]].copy()
    chosen = deduplicate_candidates(chosen)
    outputs = export_results(chosen, stem="alpha_candidates_final")

    stats = summarize_layer_stats(chosen)
    logger.info(f"final candidates: {len(chosen)}")
    return chosen, stats, outputs


print(run_unit_tests())
result_df, stats_df, output_paths = main(run_real_api=False)
print("exported:", output_paths)
result_df.head(10), stats_df

2026-04-19 21:28:21,481 | INFO | generated candidates: 24
2026-04-19 21:28:21,494 | INFO | final candidates: 1


unit tests passed
exported: {'csv': WindowsPath('D:/DBC/量化实战课_20260412/outputs/alpha_candidates_final.csv'), 'parquet': WindowsPath('D:/DBC/量化实战课_20260412/outputs/alpha_candidates_final.parquet'), 'json': WindowsPath('D:/DBC/量化实战课_20260412/outputs/alpha_candidates_final.json')}


(                                                expr                           parent  layer  heuristic       IC       IR  Turnover     mode     Score  train_score  valid_score  \
 0  group_mean(group_mean(rank(close), sector), in...  group_mean(rank(close), sector)      3      0.634  0.05817  0.75804    0.2183  dry_run  0.310471     0.316565     0.279271   
 
    test_score   oos_gap  is_overfit  test_window_years                                          expr_norm  
 0    0.340489 -0.023924       False                  2  group_mean(group_mean(rank(close),sector),indu...  ,
    layer  count  best_score  pass_rate
 0      3      1    0.310471        1.0)